## Learnning Objectives:
1. Understand overfitting and regularization
2. Know when touse Ridge vs Lasso
3. Implement and tune regularized mpdels
4. Make business recommendations

In [8]:
# Data manipulation
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Machine Learning
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.linear_model import LinearRegression, Ridge, Lasso, RidgeCV, LassoCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

# Warnings
import warnings
warnings.filterwarnings('ignore')

# Styling
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

# Random seed for reproducibility
np.random.seed(42)

print("✅ All libraries loaded successfully!")
print(f"📦 NumPy version: {np.__version__}")
print(f"📦 Pandas version: {pd.__version__}")


✅ All libraries loaded successfully!
📦 NumPy version: 2.3.4
📦 Pandas version: 2.3.3


In [9]:
# Load the dataset
df = pd.read_csv('customer_churn_data.csv')

# Quick overview
print("✅ Dataset loaded successfully!")
print(f"\n📊 Dataset Shape: {df.shape}")
print(f"   - Samples: {len(df)}")
print(f"   - Features: {len(df.columns) - 1}")  # Excluding target
print(f"   - Target: Churn_Score")

# Display first few rows
print(f"\n📋 First Few Rows:")
print(df.head())

# Basic statistics
print(f"\n📈 Dataset Summary:")
print(f"   - Missing values: {df.isnull().sum().sum()}")
print(f"   - Churn Score range: {df['Churn_Score'].min():.2f} - {df['Churn_Score'].max():.2f}")
print(f"   - Churn Score mean: {df['Churn_Score'].mean():.2f}")
print(f"   - Churn Score std: {df['Churn_Score'].std():.2f}")

# Feature categories
print(f"\n📊 Sample Features:")
print(f"   Demographics: Age, Income, Education_Level, ...")
print(f"   Account: Account_Balance, Credit_Score, ...")
print(f"   Transactions: Monthly_Transactions, Avg_Transaction_Size, ...")
print(f"   Engagement: App_Logins_Month, Customer_Service_Calls, ...")
print(f"   Products: Num_Products, Credit_Card, ...")
print(f"   Behavior: Late_Payments, Overdraft_Frequency, ...")

print(f"\n⚠️ This is a HIGH-DIMENSIONAL problem!")
print(f"   Ratio of Features to Samples: {(len(df.columns)-1)/len(df):.1%}")


✅ Dataset loaded successfully!

📊 Dataset Shape: (800, 37)
   - Samples: 800
   - Features: 36
   - Target: Churn_Score

📋 First Few Rows:
  customer_id  age        income  employment_years  education_level  \
0   CUST_0001   56  56140.195998                14                2   
1   CUST_0002   69  64729.489404                35                2   
2   CUST_0003   46  48314.984053                14                1   
3   CUST_0004   32  52403.972064                22                3   
4   CUST_0005   60  76089.350570                 5                1   

   num_dependents  account_age_months  account_balance  credit_score  \
0               0                  84     -5000.000000           802   
1               4                  99     11227.112239           654   
2               4                 111     11295.295842           524   
3               4                  66     -3043.157597           830   
4               3                  30     13953.554308           445   

 

KeyError: 'Churn_Score'

In [ ]:
# Prepare data
X = df.drop('Churn_Score', axis=1)
y = df['Churn_Score']

# Split: 70% train, 30% test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

# Scale features (important for regularization!)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("✅ Data prepared for modeling")
print(f"\n📊 Train Set: {X_train.shape}")
print(f"📊 Test Set: {X_test.shape}")


KeyError: "['Churn_Score'] not found in axis"

In [ ]:
# Fit regular linear regression (NO regularization)
lr_model = LinearRegression()
lr_model.fit(X_train_scaled, y_train)

In [ ]:
# Predictions
y_train_pred = lr_model.predict(X_train_scaled)
y_test_pred = lr_model.predict(X_test_scaled)

# Calculate metrics
train_r2 = r2_score(y_train, y_train_pred)
test_r2 = r2_score(y_test, y_test_pred)
train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))

print("="*70)
print("🔴 REGULAR LINEAR REGRESSION (NO REGULARIZATION)")
print("="*70)
print(f"\n📈 TRAINING Performance:")
print(f"   R² Score: {train_r2:.4f}")
print(f"   RMSE: {train_rmse:.4f}")
print(f"\n📉 TESTING Performance:")
print(f"   R² Score: {test_r2:.4f}")
print(f"   RMSE: {test_rmse:.4f}")
print(f"\n⚠️ OVERFITTING CHECK:")
print(f"   R² Gap: {train_r2 - test_r2:.4f}")
print(f"   RMSE Gap: {test_rmse - train_rmse:.4f}")


if (train_r2 - test_r2) > 0.1:
    print(f"\n🚨 OVERFITTING DETECTED!")
    print(f"   The model performs {train_r2 - test_r2:.1%} better on training data.")
    print(f"   This model is 'memorizing' rather than 'learning'.")
else:
    print(f"\n✅ No significant overfitting")

In [ ]:
#ridge regression (L2 Regularization)

ridge_model = Ridge(alpha=75)
ridge_model.fit(X_train_scaled, y_train)

# Predictions
y_train_pred_ridge = ridge_model.predict(X_train_scaled)
y_test_pred_ridge = ridge_model.predict(X_test_scaled)

# Calculate metrics
train_r2_ridge = r2_score(y_train, y_train_pred_ridge)
test_r2_ridge = r2_score(y_test, y_test_pred_ridge)
train_rmse_ridge = np.sqrt(mean_squared_error(y_train, y_train_pred_ridge))
test_rmse_ridge = np.sqrt(mean_squared_error(y_test, y_test_pred_ridge))

print("="*70)
print("🔵 RIDGE REGRESSION (α = 1.0)")
print("="*70)
print(f"\n📈 TRAINING Performance:")
print(f"   R² Score: {train_r2_ridge:.4f}")
print(f"   RMSE: {train_rmse_ridge:.4f}")
print(f"\n📉 TESTING Performance:")
print(f"   R² Score: {test_r2_ridge:.4f}")
print(f"   RMSE: {test_rmse_ridge:.4f}")
print(f"\n✅ OVERFITTING CHECK:")
print(f"   R² Gap: {train_r2_ridge - test_r2_ridge:.4f}")
print(f"   RMSE Gap: {test_rmse_ridge - train_rmse_ridge:.4f}")

# Compare with regular regression
print(f"\n📊 IMPROVEMENT vs Regular Regression:")
print(f"   Test R² Improvement: {test_r2_ridge - test_r2:+.4f}")
print(f"   Test RMSE Improvement: {test_rmse - test_rmse_ridge:+.4f}")
print(f"   Overfitting Reduction: {(train_r2 - test_r2) - (train_r2_ridge - test_r2_ridge):.4f}")

if (train_r2_ridge - test_r2_ridge) > 0.1:
    print(f"\n🚨 OVERFITTING DETECTED (RIDGE)!")
    print(f"   The model performs {(train_r2_ridge - test_r2_ridge):.1%} better on training data.")
    print(f"   Even with regularisation, some memorization remains.")
else:
    print(f"\n✅ No significant overfitting (RIDGE)")

In [ ]:
lasso_model = Lasso(alpha=0.1, max_iter=10000)
lasso_model.fit(X_train_scaled, y_train)

# Predictions
y_train_pred_lasso = lasso_model.predict(X_train_scaled)
y_test_pred_lasso = lasso_model.predict(X_test_scaled)

# Calculate metrics
train_r2_lasso = r2_score(y_train, y_train_pred_lasso)
test_r2_lasso = r2_score(y_test, y_test_pred_lasso)
train_rmse_lasso = np.sqrt(mean_squared_error(y_train, y_train_pred_lasso))
test_rmse_lasso = np.sqrt(mean_squared_error(y_test, y_test_pred_lasso))

# Count non-zero coefficients
n_nonzero = np.sum(lasso_model.coef_ != 0)
n_features = X_train.shape[1] # Define n_features here

print("="*70)
print("🔴 LASSO REGRESSION (α = 0.1)")
print("="*70)
print(f"\n📈 TRAINING Performance:")
print(f"   R² Score: {train_r2_lasso:.4f}")
print(f"   RMSE: {train_rmse_lasso:.4f}")
print(f"\n📉 TESTING Performance:")
print(f"   R² Score: {test_r2_lasso:.4f}")
print(f"   RMSE: {test_rmse_lasso:.4f}")
print(f"\n✂️ FEATURE SELECTION:")
print(f"   Features selected: {n_nonzero} out of {n_features}")
print(f"   Features eliminated: {n_features - n_nonzero}")
print(f"   Sparsity: {(1 - n_nonzero/n_features)*100:.1f}%")

In [ ]:
START: Regression Problem
   │
   ├─❓ Do you have MORE FEATURES than SAMPLES? (p > n)
   │   ├─ YES → ⚠️ High-dimensional problem
   │   │         ├─ Need interpretability? → 🔴 LASSO
   │   │         └─ Don't need interpretability? → 🟣 ELASTIC NET
   │   └─ NO → Continue...
   │
   ├─❓ Are your features HIGHLY CORRELATED?
   │   ├─ YES → 🔵 RIDGE (handles multicollinearity)
   │   └─ NO → Continue...
   │
   ├─❓ Do you need AUTOMATIC FEATURE SELECTION?
   │   ├─ YES → 🔴 LASSO (sets coefficients to zero)
   │   └─ NO → Continue...
   │
   ├─❓ Do you believe MANY features contribute?
   │   ├─ YES → 🔵 RIDGE (keeps all features)
   │   └─ NO → Continue...
   │
   ├─❓ Is MODEL INTERPRETABILITY critical?
   │   ├─ YES → 🔴 LASSO (fewer features = easier to explain)
   │   └─ NO → Continue...
   │
   └─✅ DECISION MADE!


In [ ]:
#find optimal value of Alpha in case of Ridge

alphas = np.logspace(-2,3,50)

train_scores = []
test_scores = []

for alpha in alphas:
    ridge_temp = Ridge(alpha=alpha)
    ridge_temp.fit(X_train_scaled, y_train)
    train_scores.append(ridge_temp.score(X_train_scaled, y_train))
    test_scores.append(ridge_temp.score(X_test_scaled, y_test))

# Best alpha
best_idx = np.argmax(test_scores)
best_alpha = alphas[best_idx]
best_score = test_scores[best_idx]

print(f"🎯 OPTIMAL ALPHA: {best_alpha:.4f}")
print(f"   Test R² at Best: {best_score:.4f}")

# Plot
plt.figure(figsize=(10, 6))
plt.plot(alphas, train_scores, label='Training R²', linewidth=2)
plt.plot(alphas, test_scores, label='Test R²', linewidth=2)
plt.axvline(x=best_alpha, color='red', linestyle='--', linewidth=2,
            label=f'Best α = {best_alpha:.2f}')
plt.xscale('log')
plt.xlabel('Alpha')
plt.ylabel('R² Score')
plt.title('Ridge Regularization Path', fontweight='bold')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

In [ ]:
# Alpha search
alphas_lasso = np.logspace(-2, 2, 50)
test_scores_lasso = []
n_features_list = []

for alpha in alphas_lasso:
    lasso_temp = Lasso(alpha=alpha, max_iter=10000)
    lasso_temp.fit(X_train_scaled, y_train)
    test_scores_lasso.append(lasso_temp.score(X_test_scaled, y_test))
    n_features_list.append(np.sum(lasso_temp.coef_ != 0))

best_idx_lasso = np.argmax(test_scores_lasso)
best_alpha_lasso = alphas_lasso[best_idx_lasso]
best_n_features = n_features_list[best_idx_lasso]

print(f"🎯 OPTIMAL LASSO ALPHA: {best_alpha_lasso:.4f}")
print(f"   Test R²: {test_scores_lasso[best_idx_lasso]:.4f}")
print(f"   Features Selected: {best_n_features}")

# Plot feature selection path
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(alphas_lasso, test_scores_lasso, linewidth=2, color='purple')
axes[0].axvline(x=best_alpha_lasso, color='red', linestyle='--', linewidth=2)
axes[0].set_xscale('log')
axes[0].set_xlabel('Alpha')
axes[0].set_ylabel('Test R²')
axes[0].set_title('Lasso Performance vs Alpha', fontweight='bold')
axes[0].grid(alpha=0.3)

axes[1].plot(alphas_lasso, n_features_list, linewidth=2, color='green')
axes[1].axvline(x=best_alpha_lasso, color='red', linestyle='--', linewidth=2)
axes[1].axhline(y=best_n_features, color='orange', linestyle=':', linewidth=2)
axes[1].set_xscale('log')
axes[1].set_xlabel('Alpha')
axes[1].set_ylabel('Number of Features')
axes[1].set_title('Feature Selection Path\n(Sparsity Control)', fontweight='bold')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()